## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

## Read bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

## Silver Transformation

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Cleaning dates

In [0]:
df = (
    df.withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == "0") | (F.length(col("sls_order_dt")) != 8),
            None
        ).otherwise(
            F.to_date(col("sls_order_dt"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == "0") | (F.length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(
            F.to_date(col("sls_ship_dt"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == "0") | (F.length(col("sls_due_dt")) != 8),
            None
        ).otherwise(
            F.to_date(col("sls_due_dt"), "yyyyMMdd")
        )
    )
)

### Sales and price correction

In [0]:
df = (
    df.withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") < 0),
            F.when(
                col("sls_quantity") != 0, 
                   col("sls_sales") / col("sls_quantity")
                   ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

### Rename columns


In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Sanity check of dataframe

In [0]:
df.limit(10).display()

order_number,product_key,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578,1,3578.0
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699,1,699.0
SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578,1,3578.0
SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578,1,3578.0
SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375,1,3375.0
SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400,1,3400.0
SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578,1,3578.0


### Writing silver tables

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

### Sanity check of silver table

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_sales
LIMIT 10

order_number,product_key,customer_id,order_date,ship_date,due_date,sales_amount,quantity,price
SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578,1,3578.0
SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699,1,699.0
SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400,1,3400.0
SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578,1,3578.0
SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578,1,3578.0
SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375,1,3375.0
SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400,1,3400.0
SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578,1,3578.0
